# Speech Emotion Router (SER): Baseline Modeling & Benchmark Validation

In the previous phase ([`01_signal_eda_features.ipynb`](./01_signal_eda_features.ipynb)), acoustic audits and cross-speaker MFCC comparisons established that physical anatomy and pitch baseline disparities justify a two-stage hierarchical model (Stage 1: Gender $\rightarrow$ Stage 2: Emotion). 

However, before implementing that phased pipeline, we must empirically put that hypothesis to the test. This phase sets our benchmark performance floor by evaluating two initial baselines against a composite target variable (`gender_emotion`), framing the task as a flat 16-class multi-classification problem (e.g., `M_angry`, `F_fearful`, `M_neutral`).

**Baseline Scope & Experimental Design:**
- **Zero-Rule Benchmark (Dummy Classifier):** Establish the absolute mathematical lower bound using uniform and stratified dummy strategies to gauge random chance across 16 imbalanced classes.
- **Linear Benchmark (Monolithic Logistic Regression):** Fit a flat Multinomial Logistic Regression pipeline with `StandardScaler` to test whether the 26 MFCC summary statistics ($\mu, \sigma$) are linearly separable across joint demographic and affective targets.
- **Strict Leakage Prevention:** Evaluate all baselines strictly via `StratifiedGroupKFold` ($k=5$) grouped by `actor_id`, ensuring no speaker's vocal tract anatomy appears in both training and test folds.
- **Hypothesis Diagnostics:** Assess macro $F_1$-score, class-level precision/recall, and a $16 \times 16$ normalized confusion matrix to pinpoint where a single flat model fails—specifically measuring whether errors stem from true emotional ambiguity or cross-gender acoustic interference.

In [33]:
from pathlib import Path

# Data handling and numerical arrays
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing & pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scikit-learn: model selection & evaluation
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Scikit-learn: baseline classifiers
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Set visualization style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

In [34]:
# Define paths and import the processed feature dataset
project_root = Path.cwd().resolve().parent
data_processed_path = project_root / "data" / "processed"

## Loading & Preparing the Data

In [35]:
df = pd.read_parquet(data_processed_path / "ser_features.parquet")

# Verify loaded shape and integrity
print(f"Loaded shape: {df.shape}")  # Expect (1440, 32)
df.info()

Loaded shape: (1440, 32)
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 32 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   file_path     1440 non-null   str    
 1   actor_id      1440 non-null   int64  
 2   gender        1440 non-null   str    
 3   emotion       1440 non-null   str    
 4   intensity     1440 non-null   str    
 5   repetition    1440 non-null   int64  
 6   mfcc_mean_0   1440 non-null   float64
 7   mfcc_std_0    1440 non-null   float64
 8   mfcc_mean_1   1440 non-null   float64
 9   mfcc_std_1    1440 non-null   float64
 10  mfcc_mean_2   1440 non-null   float64
 11  mfcc_std_2    1440 non-null   float64
 12  mfcc_mean_3   1440 non-null   float64
 13  mfcc_std_3    1440 non-null   float64
 14  mfcc_mean_4   1440 non-null   float64
 15  mfcc_std_4    1440 non-null   float64
 16  mfcc_mean_5   1440 non-null   float64
 17  mfcc_std_5    1440 non-null   float64
 18  mfcc_mean_6   

In [36]:
# Construct the composite target label for monolithic classification
df["gender_emotion"] = df["gender"] + "_" + df["emotion"]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 33 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   file_path       1440 non-null   str    
 1   actor_id        1440 non-null   int64  
 2   gender          1440 non-null   str    
 3   emotion         1440 non-null   str    
 4   intensity       1440 non-null   str    
 5   repetition      1440 non-null   int64  
 6   mfcc_mean_0     1440 non-null   float64
 7   mfcc_std_0      1440 non-null   float64
 8   mfcc_mean_1     1440 non-null   float64
 9   mfcc_std_1      1440 non-null   float64
 10  mfcc_mean_2     1440 non-null   float64
 11  mfcc_std_2      1440 non-null   float64
 12  mfcc_mean_3     1440 non-null   float64
 13  mfcc_std_3      1440 non-null   float64
 14  mfcc_mean_4     1440 non-null   float64
 15  mfcc_std_4      1440 non-null   float64
 16  mfcc_mean_5     1440 non-null   float64
 17  mfcc_std_5      1440 non-null   float64
 18 

In [37]:
# Validate - Inspect class counts and normalized distributions
target_counts = pd.DataFrame({
    "count": df["gender_emotion"].value_counts(),
    "proportion": df["gender_emotion"].value_counts(normalize=True).round(4)
})

display(target_counts)
print(f"Total classes: {df['gender_emotion'].nunique()}")

,count,proportion
gender_emotion,,
M_disgust,96,0.0667
F_angry,96,0.0667
M_fearful,96,0.0667
F_disgust,96,0.0667
M_angry,96,0.0667
M_calm,96,0.0667
F_calm,96,0.0667
M_sad,96,0.0667
F_sad,96,0.0667


Total classes: 16


This is as expected - there are 48 `normal` for all emotions and 48 `strong` for all emotions except `neutral`.

We can now drop some of the old features:
- `file_path`, `repetition`, `intensity` - metadata columns, no value for modeling 
- `gender`, `emotion` - these are now superseded by the new target variable


In [38]:
cols_to_drop = ["file_path", "gender", "intensity", "emotion", "repetition"]
df = df.drop(columns=cols_to_drop)

# Verify schema retention
print(f"Remaining shape: {df.shape}")
df.info()

Remaining shape: (1440, 28)
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   actor_id        1440 non-null   int64  
 1   mfcc_mean_0     1440 non-null   float64
 2   mfcc_std_0      1440 non-null   float64
 3   mfcc_mean_1     1440 non-null   float64
 4   mfcc_std_1      1440 non-null   float64
 5   mfcc_mean_2     1440 non-null   float64
 6   mfcc_std_2      1440 non-null   float64
 7   mfcc_mean_3     1440 non-null   float64
 8   mfcc_std_3      1440 non-null   float64
 9   mfcc_mean_4     1440 non-null   float64
 10  mfcc_std_4      1440 non-null   float64
 11  mfcc_mean_5     1440 non-null   float64
 12  mfcc_std_5      1440 non-null   float64
 13  mfcc_mean_6     1440 non-null   float64
 14  mfcc_std_6      1440 non-null   float64
 15  mfcc_mean_7     1440 non-null   float64
 16  mfcc_std_7      1440 non-null   float64
 17  mfcc_mean_8     

We keep `actor_id` for the moment - while it will not be used for training, we will need it to develop our splits.

### Develop Training/Test Splits

In [39]:
# Extract speaker groups for StratifiedGroupKFold before dropping
groups = df["actor_id"].copy()

# NOW we can drop actor_id safely
df = df.drop(columns=["actor_id"])

X = df.drop(columns=["gender_emotion"]).copy()
y = df["gender_emotion"].copy()

# Audit schema and memory footprint using built-in .info()
print(X.shape)
X.info()

(1440, 26)
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mfcc_mean_0   1440 non-null   float64
 1   mfcc_std_0    1440 non-null   float64
 2   mfcc_mean_1   1440 non-null   float64
 3   mfcc_std_1    1440 non-null   float64
 4   mfcc_mean_2   1440 non-null   float64
 5   mfcc_std_2    1440 non-null   float64
 6   mfcc_mean_3   1440 non-null   float64
 7   mfcc_std_3    1440 non-null   float64
 8   mfcc_mean_4   1440 non-null   float64
 9   mfcc_std_4    1440 non-null   float64
 10  mfcc_mean_5   1440 non-null   float64
 11  mfcc_std_5    1440 non-null   float64
 12  mfcc_mean_6   1440 non-null   float64
 13  mfcc_std_6    1440 non-null   float64
 14  mfcc_mean_7   1440 non-null   float64
 15  mfcc_std_7    1440 non-null   float64
 16  mfcc_mean_8   1440 non-null   float64
 17  mfcc_std_8    1440 non-null   float64
 18  mfcc_mean_9   1440 non-null 

In [40]:
# # Save intermediates...
# x_path = data_processed_path / "02_1_X_full.parquet"
# y_path = data_processed_path / "02_1_Y_full.parquet"

# # Export X and y to Parquet
# X.to_parquet(x_path, index=False)
# y.to_frame(name="gender_emotion").to_parquet(y_path, index=False)

# # Verify artifacts written to disk
# print(f"X saved to: {x_path} (exists: {x_path.exists()})")
# print(f"y saved to: {y_path} (exists: {y_path.exists()})")

In [43]:
# # Audit saved artifacts using .info()

# x_path = data_processed_path / "02_1_X_full.parquet"
# y_path = data_processed_path / "02_1_Y_full.parquet"
# print("--- X Artifact ---")
# pd.read_parquet(x_path).info()  # Expected: (1440, 26)

# print("\n--- y Artifact ---")
# pd.read_parquet(y_path).info()  # Expected: Just the gender_emotion

## Baseline 1: DummyClassifier

## Baseline 2: Monolithic Logistic Regression